<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-02-embeddings-and-vectors/lesson-2.4-alloydb-bigquery/practice/GCP_Capstone_2.4_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 2.4 — AlloyDB pgvector & BigQuery

8 exercises with solutions. Master pgvector operators, ScaNN indexing, VECTOR_SEARCH(), and the decision framework.

Runnable companion to the published practice lab. Exercises 1, 2, 3 and 5 are **reference SQL — run them in the BigQuery / AlloyDB console**, not in Colab. Exercises 4, 6, 7 and 8 are runnable Python: run the setup cell first, then each solution cell.

---

In [ ]:
!pip install -q "google-cloud-alloydb-connector[pg8000]" sqlalchemy google-genai==2.21.0

from google.colab import auth  # Colab only
auth.authenticate_user()  # Application Default Credentials, not API keys

# Exercises 4, 6, 7 and 8 are runnable and each sets its own PROJECT_ID at the top -
# change it there. Exercises 1, 2, 3 and 5 are reference SQL for the console.
print('Setup done.')


## Exercise 1: pgvector Distance Operators  
**Difficulty:** Easy

Write SQL for cosine, L2, and inner product search using the three pgvector operators.

1. Write a cosine distance query with <=>
2. Write an L2 distance query with <->
3. Write an inner product query with <#>

**Solution:**

*Reference SQL — run this in the AlloyDB / BigQuery console, not in Colab.*

```sql
-- Cosine distance (DEFAULT for text embeddings)
SELECT content, embedding <=> '[0.1,0.2,...]'::vector AS cosine_dist
FROM documents ORDER BY cosine_dist LIMIT 5;

-- L2 / Euclidean distance
SELECT content, embedding <-> '[0.1,0.2,...]'::vector AS l2_dist
FROM documents ORDER BY l2_dist LIMIT 5;

-- Negative inner product (requires normalized vectors)
SELECT content, embedding <#> '[0.1,0.2,...]'::vector AS neg_ip
FROM documents ORDER BY neg_ip LIMIT 5;
```

## Exercise 2: Compare 3 Index Types  
**Difficulty:** Easy

Write CREATE INDEX statements for IVFFlat, HNSW, and ScaNN on the same table.

1. Create IVFFlat index with 100 lists
2. Create HNSW index with m=16, ef_construction=64
3. Create ScaNN index with 100 leaves

**Solution:**

*Reference SQL — run this in the AlloyDB / BigQuery console, not in Colab.*

```sql
-- IVFFlat (any PostgreSQL)
CREATE INDEX idx_ivf ON documents
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);

-- HNSW (any PostgreSQL)
CREATE INDEX idx_hnsw ON documents
USING hnsw (embedding vector_cosine_ops)
WITH (m = 16, ef_construction = 64);

-- ScaNN (AlloyDB only!)
CREATE EXTENSION IF NOT EXISTS alloydb_scann CASCADE;
CREATE INDEX idx_scann ON documents
USING scann (embedding cosine)
WITH (num_leaves = 100);
ANALYZE documents;
```

## Exercise 3: BigQuery VECTOR_SEARCH  
**Difficulty:** Easy

Write a complete VECTOR_SEARCH query with inline AI.GENERATE_EMBEDDING for the query.

1. Reference a base table with embeddings
2. Generate query embedding inline
3. Return top 5 results with cosine distance

**Solution:**

*Reference SQL — run this in the AlloyDB / BigQuery console, not in Colab.*

```sql
SELECT base.title, base.id, distance
FROM VECTOR_SEARCH(
  TABLE `my_dataset.doc_embeddings`,
  'ml_generate_embedding_result',
  (SELECT ml_generate_embedding_result AS embedding
   FROM AI.GENERATE_EMBEDDING(
     MODEL `my_dataset.embed_model`,
     (SELECT 'What is vector search?' AS content),
     STRUCT(768 AS output_dimensionality, 'RETRIEVAL_QUERY' AS task_type,
            TRUE AS flatten_json_output)
   )),
  top_k => 5,
  distance_type => 'COSINE'
);
```

## Exercise 4: AlloyDB + Python Pipeline  
**Difficulty:** Medium

Write Python code to connect to AlloyDB, insert 5 documents with embeddings, and query.

1. Connect with the AlloyDB Language Connector (IAM auth, no password)
2. Generate embeddings with google-genai (one text per request)
3. Insert 5 docs into the vector column
4. Query with <=> cosine distance

**Solution:**

In [ ]:
# Exercise 4 - AlloyDB from Python via the Language Connector (IAM auth, no password).
# pip install "google-cloud-alloydb-connector[pg8000]" sqlalchemy google-genai==2.21.0
import sqlalchemy
from google.cloud.alloydb.connector import Connector
from google import genai
from google.genai import types

PROJECT_ID = "documind-ai-YOUR-ID"          # CHANGE THIS
INSTANCE_URI = (f"projects/{PROJECT_ID}/locations/us-central1"
                "/clusters/documind/instances/documind-primary")
# DB_USER must equal the identity behind your ADC: your own email in Colab after
# auth.authenticate_user(), sa-name@PROJECT.iam when running as a service account.
DB_USER = f"documind-app@{PROJECT_ID}.iam"  # IAM principal - no password anywhere
DB_NAME = "documind"

DOCS = [
    "AlloyDB provides sub-millisecond vector search.",
    "ScaNN is an AlloyDB-exclusive vector index.",
    "pgvector adds a vector column type to PostgreSQL.",
    "Firestore find_nearest() is DocuMind's prototype store.",
    "BigQuery VECTOR_SEARCH is batch-optimised, not real-time.",
]


def make_engine():
    """AlloyDB is private-IP by default: unreachable from Colab without the connector."""
    connector = Connector()

    def getconn():
        return connector.connect(
            INSTANCE_URI, "pg8000",
            user=DB_USER, db=DB_NAME, enable_iam_auth=True,
        )

    return sqlalchemy.create_engine("postgresql+pg8000://", creator=getconn)


def embed(client, text, task_type):
    """gemini-embedding-001 takes ONE text per request on Vertex AI - loop, never a list."""
    return client.models.embed_content(
        model="gemini-embedding-001", contents=text,
        config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=768),
    ).embeddings[0].values


def run_pipeline():
    # Embeddings are regional-only -> us-central1. Gemini 3.x generation would need a
    # SECOND client on location="global".
    ai = genai.Client(enterprise=True, project=PROJECT_ID, location="us-central1")
    engine = make_engine()
    with engine.connect() as db:
        for text in DOCS:
            db.execute(
                sqlalchemy.text("INSERT INTO documents (content, embedding) "
                                "VALUES (:c, CAST(:e AS vector))"),
                {"c": text, "e": str(embed(ai, text, "RETRIEVAL_DOCUMENT"))})
        db.commit()
        q = str(embed(ai, "How fast is AlloyDB for vector queries?", "RETRIEVAL_QUERY"))
        rows = db.execute(sqlalchemy.text("""
            SELECT content, embedding <=> CAST(:q AS vector) AS distance
            FROM documents
            ORDER BY embedding <=> CAST(:q AS vector) LIMIT 5
        """), {"q": q}).fetchall()
    for content, dist in rows:
        print(f"  [{1 - dist:.4f}] {content}")


if PROJECT_ID == "documind-ai-YOUR-ID":
    print("Set PROJECT_ID and provision an AlloyDB cluster, then call run_pipeline().")
    print(f"Prepared {len(DOCS)} documents. Connector target:\n  {INSTANCE_URI}")
    print(f"Signing in as IAM principal {DB_USER} - no password is stored anywhere.")
else:
    run_pipeline()

## Exercise 5: Filtered SQL Vector Search  
**Difficulty:** Medium

Write a query combining WHERE + JOIN + vector ORDER BY in one statement.

1. Filter by category and date range
2. JOIN with authors table
3. Order by cosine distance to query vector

**Solution:**

*Reference SQL — run this in the AlloyDB / BigQuery console, not in Colab.*

```sql
-- The corpus is embedded with gemini-embedding-001, so the query embeds with it too.
-- documind-embed-768 is gemini-embedding-001 registered with a 768-d input transform;
-- the pre-registered gemini-embedding-001 id returns 3072-d and vector(768) rejects it.
SELECT d.content, a.name AS author,
       d.embedding <=> embedding('documind-embed-768',
           'How does attention work?')::vector AS distance
FROM documents d
JOIN authors a ON d.author_id = a.id
WHERE d.category = 'ai_ml'
  AND d.created_at > '2025-01-01'
ORDER BY d.embedding <=> embedding('documind-embed-768',
    'How does attention work?')::vector
LIMIT 5;
```

## Exercise 6: BigQuery Batch Embedding  
**Difficulty:** Medium

Use AI.GENERATE_EMBEDDING to embed rows from a table, create an IVF index, and search.

1. Create the CLOUD_RESOURCE connection to Vertex AI and grant it roles/aiplatform.user
2. Register gemini-embedding-001 with CREATE MODEL
3. Batch-embed with AI.GENERATE_EMBEDDING
4. Create the IVF index (needs a base table of at least 10 MB) and run VECTOR_SEARCH

**Solution:**

In [ ]:
# Exercise 6 - BigQuery batch embedding. The SQL runs in BigQuery; this cell builds it
# and states the two prerequisites, so the cell itself needs no warehouse.
PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS
DATASET = "documind_vectors"
CONNECTION = f"{PROJECT_ID}.us.vertex_conn"

PREREQUISITES = [
    "A BigQuery -> Vertex AI connection (the model call has no other way out):\n"
    "     bq mk --connection --location=us --connection_type=CLOUD_RESOURCE vertex_conn\n"
    "   then grant that connection's service account roles/aiplatform.user.",
    "A base table of at least 10 MB - below that CREATE VECTOR INDEX succeeds but\n"
    "   the index is never populated (coverage 0%) and VECTOR_SEARCH runs brute force;\n"
    "   the topic notebook's 3-row demo table is far under it.",
]


def bq_workflow_sql(dataset=DATASET, connection=CONNECTION, top_k=5):
    """Return the four-statement BigQuery workflow for the DocuMind corpus."""
    return f"""
-- 1. Remote embedding model: the SAME model this store is written with.
--    gemini-embedding-001 returns 3072-d by default; request 768-d output (MRL
--    truncation) or BigQuery stores 3072-d vectors that no longer match the 768-d
--    store contract.
CREATE OR REPLACE MODEL `{dataset}.embed_model`
  REMOTE WITH CONNECTION `{connection}`
  OPTIONS (ENDPOINT = 'gemini-embedding-001');

-- 2. Batch-embed the corpus. The STRUCT pins the output to 768 dimensions, which
--    is the store's contract; without it BigQuery stores 3072-d vectors.
CREATE OR REPLACE TABLE `{dataset}.doc_embeddings` AS
SELECT * FROM AI.GENERATE_EMBEDDING(
  MODEL `{dataset}.embed_model`,
  (SELECT id, body AS content FROM `{dataset}.documents`),
  STRUCT(768 AS output_dimensionality, 'RETRIEVAL_DOCUMENT' AS task_type,
         TRUE AS flatten_json_output));

-- 3. Vector index. Needs a base table of at least 10 MB or it is never populated
--    (coverage 0%). Not available in the Standard edition (Enterprise or higher);
--    verify on the BigQuery pricing page before you quote a cost.
CREATE OR REPLACE VECTOR INDEX doc_idx
ON `{dataset}.doc_embeddings`(ml_generate_embedding_result)
OPTIONS (index_type = 'IVF', distance_type = 'COSINE');

-- 4. Search
SELECT base.id, base.content, distance
FROM VECTOR_SEARCH(
  TABLE `{dataset}.doc_embeddings`,
  'ml_generate_embedding_result',
  (SELECT ml_generate_embedding_result AS embedding
   FROM AI.GENERATE_EMBEDDING(
     MODEL `{dataset}.embed_model`,
     (SELECT 'How does attention work?' AS content),
     STRUCT(768 AS output_dimensionality, 'RETRIEVAL_QUERY' AS task_type,
            TRUE AS flatten_json_output))),
  top_k => {top_k}, distance_type => 'COSINE');
"""


print("Two prerequisites before ANY of this SQL runs:")
for i, prereq in enumerate(PREREQUISITES, start=1):
    print(f"  {i}. {prereq}")
print(bq_workflow_sql())

## Exercise 7: Decision Framework Quiz  
**Difficulty:** Challenge

For each scenario, pick the right vector DB and justify your choice.

1. 500-doc prototype, zero budget
2. 50K docs, real-time chatbot, needs JOINs
3. 10M products, nightly batch recommendations
4. 1K docs, need ACID transactions
5. Data already in BigQuery, occasional similarity queries

**Solution:**

In [ ]:
# Exercise 7 - the decision framework, as code you can re-run on your own scenarios.
USD_INR = 85  # rate as of 2026-09-03

ALLOYDB_USD = 200        # minimum production config: 2 vCPU / 16 GiB, no read pool
FIRESTORE_100K_USD = 15  # storage + reads for the DocuMind workload at 100K docs


def bq_usd(num_docs):
    """BigQuery: ~$10 per 1M docs (storage + a small query allowance, Step 9 table)."""
    return round(10 * num_docs / 1_000_000)


def recommend_vector_db(num_docs, need_realtime=True, need_sql=False, budget_inr=0):
    """Return (choice, why, usd_per_month) for one DocuMind scenario."""
    if num_docs < 10_000 and not need_sql:
        return ("Firestore", "free tier, flat exact KNN via find_nearest()", 0)
    if need_realtime and (need_sql or num_docs >= 10_000):
        if budget_inr and budget_inr < ALLOYDB_USD * USD_INR:
            return ("Firestore (budget-capped)",
                    "AlloyDB's minimum config is over budget; accept the latency", 0)
        return ("AlloyDB + ScaNN", "pgvector <=> at 1-30ms with full SQL WHERE/JOIN",
                ALLOYDB_USD)
    return ("BigQuery VECTOR_SEARCH", "serverless batch; seconds of latency is fine",
            bq_usd(num_docs))


SCENARIOS = [
    ("500-doc prototype, zero budget",             500, True, False, 0),
    ("50K docs, real-time chatbot, needs JOINs", 50_000, True, True, 50_000),
    ("10M products, nightly batch recs",     10_000_000, False, False, 0),
    ("1K docs, ACID transactions required",       1_000, True, True, 50_000),
    ("Already in BigQuery, occasional queries", 2_000_000, False, True, 0),
]

print(f"{'Scenario':<42}{'Choice':<26}{'~$/mo':>7}{'~Rs/mo':>10}")
print("-" * 85)
for label, docs, realtime, sql, budget in SCENARIOS:
    choice, why, usd = recommend_vector_db(docs, realtime, sql, budget)
    print(f"{label:<42}{choice:<26}{usd:>7}{usd * USD_INR:>10,}")
    print(f"{'':<42}{why}")

# Cost comparison - one AlloyDB figure and one Firestore figure for the whole module.
MONTHLY_INR = {
    # The 1M cell is READ-VOLUME driven, not document-count driven: it assumes a
    # busier query load, so it is not on the same basis as the canonical $15 at 100K.
    "Firestore": {10_000: 0, 100_000: FIRESTORE_100K_USD * USD_INR, 1_000_000: 5_000},
    "AlloyDB": {n: ALLOYDB_USD * USD_INR for n in (10_000, 100_000, 1_000_000)},
    "BigQuery": {10_000: 0, 100_000: 170, 1_000_000: 850},  # storage + a small query allowance
}

sizes = (10_000, 100_000, 1_000_000)
print(f"\n{'Service':<12}" + "".join(f"{f'{n:,} docs':>22}" for n in sizes))
print("-" * 78)
for service, by_size in MONTHLY_INR.items():
    cells = "".join(f"{f'Rs {by_size[n]:,} (${round(by_size[n] / USD_INR):,})':>22}"
                    for n in sizes)
    print(f"{service:<12}{cells}")

print("\nAlloyDB is one flat figure: the minimum production config is about $200/mo")
print("= Rs 17,000, whatever the corpus size. Firestore at 100K DocuMind docs is")
print("about $15/mo = Rs 1,275. Storage is not the driver - 10,000 documents with")
print("768-d embeddings are only about 80 MB (10K x 8 KB).")
print("\nTwo different bases, so do not read across the rows: Firestore and AlloyDB")
print("are storage + serving; the BigQuery row is storage + a small query allowance,")
print("because BigQuery bills queries separately as on-demand data scanned. Query")
print("cost moves with usage, not with corpus size.")
print("\nThe Firestore 1M figure is read-volume driven, not document-count driven,")
print("which is why it is not ten times the 100K figure.")

## Exercise 8: DocuMind Architecture Document  
**Difficulty:** Challenge

Write a 1-page architecture document for DocuMind specifying the vector database strategy.

1. Describe the Firestore prototype phase
2. Plan the AlloyDB production migration
3. Define BigQuery analytics integration
4. Specify embedding model, dimensions, task types

**Solution:**

In [ ]:
# Exercise 8 - print the template, fill in every blank, ship it with your PR.
ARCH_DOC_TEMPLATE = """
# DocuMind Vector Architecture - [your name], [date]

## 1. Embedding contract (fill in once, then never break it)
- Model: gemini-embedding-001        Dimensions: 768
- Task types: RETRIEVAL_DOCUMENT on write, RETRIEVAL_QUERY on read
- Batch limit: ONE text per embed_content request on Vertex AI (loop, never a list)
- ONE MODEL PER STORE: a store written with this model is read with this model.
  Module 4's managed Vector Search index is a SEPARATE store on text-embedding-005
  (verified on the Vertex model page 2026-09-03; re-check before relying on it),
  so moving a corpus between them re-embeds it - it never mixes models in one index.
- Clients: embeddings are regional (us-central1); Gemini 3.x generation is global.

## 2. Phase 1 - prototype ([___] docs)
- Store: Firestore (default) database in asia-south1 (Mumbai; the location is
  permanent and cannot be changed after creation)
- Query: find_nearest(), COSINE, flat exact KNN
- Cost: Rs [___]/mo          Latency budget: [___] ms

## 3. Phase 2 - production ([___] docs)
- Store: AlloyDB, pgvector <=> with a ScaNN index (mode = 'AUTO')
- Access: AlloyDB Language Connector, enable_iam_auth=True, no password in code
- Auto-embedding: GENERATED ALWAYS AS (embedding('documind-embed-768', content))
  documind-embed-768 = gemini-embedding-001 registered under our own model_id with a
  custom input transform that pins outputDimensionality: 768 (the pre-registered
  gemini-embedding-001 id returns 3072-d).
- Cost: about $200/mo = Rs 17,000 (minimum config, USD_INR = 85)
- Latency budget: [___] ms

## 4. Phase 3 - analytics
- Store: BigQuery, AI.GENERATE_EMBEDDING + VECTOR_SEARCH
- Prerequisites: a CLOUD_RESOURCE connection to Vertex AI; a base table of at least
  10 MB, or CREATE VECTOR INDEX succeeds but the index is never populated
- Cost basis: storage is small; queries bill separately per TiB scanned

## 5. Migration trigger and rollback
- We move from Phase 1 to Phase 2 when: [___]
- Re-embedding plan (documents, hours, cost): [___]
- Rollback if Phase 2 regresses: [___]
"""

print(ARCH_DOC_TEMPLATE)
print(f"Template length: {len(ARCH_DOC_TEMPLATE.splitlines())} lines, "
      f"{ARCH_DOC_TEMPLATE.count('[___]')} blanks to fill.")